In [5]:
import json
from typing import Dict, List, Any
import os
import datetime

def generate_qa_dataset(chebi_reasoner, output_file="chebi_qa_dataset.json", max_entities=None):
    """
    Generate a comprehensive QA dataset from the ChEBI ontology using the provided reasoner.
    
    Args:
        chebi_reasoner: An initialized ChEBIReasoner instance
        output_file: Path to save the generated dataset
        max_entities: Maximum number of entities to process (for testing)
        
    Returns:
        Dictionary containing the generated dataset
    """
    print(f"Generating QA dataset from ChEBI ontology...")
    
    # Initialize dataset structure
    dataset = {
        "metadata": {
            "description": "ChEBI Ontology Question-Answer Dataset",
            "created_on": datetime.datetime.now().isoformat(),
            "ontology_source": chebi_reasoner.onto.base_iri,
            "num_entities": len(chebi_reasoner.entity_cache)
        },
        "qa_pairs": [],
        "entity_index": {}  # Maps entities to their QA pairs
    }
    
    # Statistics tracking
    stats = {
        "total_entities_processed": 0,
        "total_qa_pairs": 0,
        "entities_with_qa_pairs": 0,
        "qa_pairs_by_reasoning_type": {
            "basic": 0,
            "comparative": 0,
            "multi_step": 0,
            "structural": 0
        },
        "qa_pairs_by_reasoning_subtype": {}
    }
    
    # Limit number of entities if specified
    entity_items = list(chebi_reasoner.entity_cache.items())
    if max_entities:
        entity_items = entity_items[:max_entities]
    
    # Process each entity
    print(f"Processing {len(entity_items)} entities...")
    
    for i, (chebi_id, entity_obj) in enumerate(entity_items):
        # Skip entities without names
        if not hasattr(entity_obj, "name") or not entity_obj.name:
            continue
            
        entity_name = entity_obj.name
        
        # Update statistics
        stats["total_entities_processed"] += 1
        
        # Progress reporting
        if i % 100 == 0:
            print(f"Processed {i}/{len(entity_items)} entities")
        
        # Get reasoning facts about this entity
        reasoning = chebi_reasoner._reason_about_entity(chebi_id, entity_name)
        
        # Skip entities with no facts
        if not reasoning.get("facts"):
            continue
        
        # Generate QA pairs for this entity
        entity_qa_pairs = generate_qa_pairs_for_entity(chebi_id, entity_name, reasoning)
        
        if entity_qa_pairs:
            # Update entity index
            dataset["entity_index"][chebi_id] = {
                "name": entity_name,
                "qa_ids": [qa["qa_id"] for qa in entity_qa_pairs]
            }
            
            # Add QA pairs to dataset
            dataset["qa_pairs"].extend(entity_qa_pairs)
            
            # Update statistics
            stats["entities_with_qa_pairs"] += 1
            stats["total_qa_pairs"] += len(entity_qa_pairs)
            
            # Count by reasoning type
            for qa in entity_qa_pairs:
                reasoning_type = qa.get("reasoning_type", "basic")
                reasoning_subtype = qa.get("reasoning_subtype", "unknown")
                
                stats["qa_pairs_by_reasoning_type"][reasoning_type] = stats["qa_pairs_by_reasoning_type"].get(reasoning_type, 0) + 1
                stats["qa_pairs_by_reasoning_subtype"][reasoning_subtype] = stats["qa_pairs_by_reasoning_subtype"].get(reasoning_subtype, 0) + 1
    
    # Add statistics to metadata
    dataset["metadata"]["statistics"] = stats
    
    # Write the dataset to a file
    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=2)
    
    print(f"✅ Generated QA dataset with {stats['total_qa_pairs']} QA pairs from {stats['entities_with_qa_pairs']} entities")
    print(f"✅ Dataset saved to {output_file}")
    
    return dataset


def generate_qa_pairs_for_entity(entity_id, entity_name, reasoning):
    """
    Generate QA pairs for a specific entity based on its reasoning information.
    This is where we'll implement different question types.
    
    Args:
        entity_id: ChEBI ID of the entity
        entity_name: Name of the entity
        reasoning: Reasoning information from the reasoner
        
    Returns:
        List of QA pair dictionaries
    """
    qa_pairs = []
    
    # Generate basic QA pairs
    basic_qa_pairs = generate_basic_qa_pairs(entity_id, entity_name, reasoning)
    qa_pairs.extend(basic_qa_pairs)
    
    # Additional types can be implemented and called here
    # qa_pairs.extend(generate_structural_qa_pairs(entity_id, entity_name, reasoning))
    
    return qa_pairs


def generate_basic_qa_pairs(entity_id, entity_name, reasoning):
    """
    Generate basic QA pairs based on direct ontology relationships.
    
    Args:
        entity_id: ChEBI ID of the entity
        entity_name: Name of the entity
        reasoning: Reasoning information from the reasoner
        
    Returns:
        List of basic QA pair dictionaries
    """
    qa_pairs = []
    relationship_descriptions = {
        "is_a": "{entity_a} is a type or subclass of {entity_b}",
        "has_part": "{entity_a} contains {entity_b} as a structural component",
        "is_conjugate_base_of": "{entity_a} is formed by the loss of a proton from {entity_b}",
        "is_conjugate_acid_of": "{entity_a} is formed by the addition of a proton to {entity_b}",
        "is_tautomer_of": "{entity_a} and {entity_b} are structural isomers that readily interconvert",
        "is_enantiomer_of": "{entity_a} is a mirror image of {entity_b} that cannot be superimposed",
        "has_functional_parent": "{entity_a} is derived from {entity_b} by functional modification",
        "has_parent_hydride": "{entity_a} is derived from {entity_b} which is its parent hydride",
        "is_substituent_group_from": "{entity_a} is a substituent group derived from {entity_b}",
        "has_role": "{entity_a} functions as a {entity_b} in biological or chemical contexts"
    }
    
    # 1. Classification questions (is_a relationship)
    if "parent_classes" in reasoning and reasoning["parent_classes"]:
        for parent in reasoning["parent_classes"]:
            question = f"Is {entity_name} a type of {parent}?"
            answer = f"Yes, {entity_name} is a type of {parent}."
            
            qa_pairs.append({
                "qa_id": f"basic_classification_{entity_id}_{hash(parent) % 10000}",
                "entity_id": entity_id,
                "entity_name": entity_name,
                "question": question,
                "answer": answer,
                "reasoning_type": "basic",
                "reasoning_subtype": "classification",
                "reasoning_steps": [
                    f"Looking up {entity_name} in the ChEBI ontology",
                    f"Found that {entity_name} has parent class {parent}",
                    f"Therefore, {entity_name} is a type of {parent}"
                ]
            })
    
    # 2. Relationship questions
    if "relationships" in reasoning and reasoning["relationships"]:
        for rel_type, targets in reasoning["relationships"].items():
            readable_rel = rel_type.replace('_', ' ')
            
            for target in targets:
                question = f"What is the relationship between {entity_name} and {target}?"
                answer = f"{entity_name} {readable_rel} {target}."
                
                rel_description = relationship_descriptions.get(rel_type, "")
                if rel_description:
                    answer += f" This means that {rel_description.format(entity_a=entity_name, entity_b=target)}."
                
                qa_pairs.append({
                    "qa_id": f"basic_relationship_{entity_id}_{hash(target) % 10000}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "related_entity": target,
                    "relationship_type": rel_type,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": "relationship",
                    "reasoning_steps": [
                        f"Looking up {entity_name} in the ChEBI ontology",
                        f"Examining relationships to other entities",
                        f"Found relationship: {entity_name} {readable_rel} {target}"
                    ]
                })
    
    return qa_pairs


def main():
    """
    Example usage of the ChEBI QA Dataset Generator.
    """
    # Import the ChEBIReasoner from your existing code
    from your_module import ChEBIReasoner
    
    # Path to ChEBI ontology file
    chebi_owl_path = "/path/to/chebi.owl"
    
    # Initialize the reasoner
    chebi_reasoner = ChEBIReasoner(chebi_owl_path)
    
    # Generate the QA dataset
    dataset = generate_qa_dataset(chebi_reasoner, "chebi_qa_dataset.json", max_entities=100)
    
    print(f"Generated {len(dataset['qa_pairs'])} QA pairs")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'your_module'

In [9]:
import sys
import os 
import json 
import gc  # For garbage collection
import time
import numpy as np
from tqdm import tqdm  # For progress tracking
sys.path.append('/home/matt/Proj/Hermeticav2/src')
from pipeline.ChEBIPipeline import ChEBIPipeline as AnnotationCheck
from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
from reasoning.LLMFeedback.Pipe2NL import extract_chebi_ids, ChEBINameRetriever, replace_chebi_ids, format_json_as_prompt
from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate
import pandas as pd


In [7]:
class ChEBIOntologyAccess:
    """
    Access layer for ChEBI ontology using the existing ChEBIReasoner.
    Provides simplified methods to access entity information and relationships.
    """
    
    def __init__(self, chebi_reasoner):
        """
        Initialize with an existing ChEBIReasoner instance.
        
        Args:
            chebi_reasoner: An initialized ChEBIReasoner instance
        """
        self.reasoner = chebi_reasoner
    
    def get_entity_info(self, entity_id):
        """
        Get basic information about an entity.
        
        Args:
            entity_id: ChEBI ID of the entity
            
        Returns:
            Dictionary with entity information or None if not found
        """
        # Get entity from the ontology
        entity_obj = self.reasoner._get_entity_by_id(entity_id)
        if not entity_obj:
            return None
            
        # Skip entities without names
        if not hasattr(entity_obj, "name") or not entity_obj.name:
            return None
            
        entity_name = entity_obj.name
        
        # Get reasoning information
        reasoning = self.reasoner._reason_about_entity(entity_id, entity_name)
        
        return {
            "id": entity_id,
            "name": entity_name,
            "entity_obj": entity_obj,
            "reasoning": reasoning
        }
    
    def get_entities_with_relationship(self, relationship_type, max_entities=100):
        """
        Find entities that have a specific relationship type.
        
        Args:
            relationship_type: Type of relationship to look for
            max_entities: Maximum number of entities to return
            
        Returns:
            List of entity dictionaries with the specified relationship
        """
        results = []
        
        for entity_id, entity_obj in self.reasoner.entity_cache.items():
            if len(results) >= max_entities:
                break
                
            # Skip entities without names
            if not hasattr(entity_obj, "name") or not entity_obj.name:
                continue
                
            entity_name = entity_obj.name
            
            # Check if entity has the relationship
            if hasattr(entity_obj, relationship_type) and getattr(entity_obj, relationship_type):
                entity_info = self.get_entity_info(entity_id)
                if entity_info:
                    results.append(entity_info)
        
        return results
    
    def get_entity_count(self):
        """
        Get the total number of entities in the ontology.
        
        Returns:
            Integer count of entities
        """
        return len(self.reasoner.entity_cache)
    
    def get_relationship_types(self):
        """
        Get all relationship types used in the ontology.
        
        Returns:
            List of relationship type strings
        """
        relationship_types = set()
        
        # Get all object properties from the ontology
        for prop in self.reasoner.onto.object_properties():
            if hasattr(prop, "name") and prop.name:
                relationship_types.add(prop.name)
        
        return sorted(list(relationship_types))


# Example usage:
def main():
    from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
    
    # Initialize the reasoner with your ontology file
    chebi_reasoner = ChEBIReasoner("/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl")
    
    # Create the access layer
    ontology_access = ChEBIOntologyAccess(chebi_reasoner)
    
    # Get information about a specific entity
    entity_info = ontology_access.get_entity_info("CHEBI:17790")  # Methanol
    if entity_info:
        print(f"Entity: {entity_info['name']} ({entity_info['id']})")
        
        # Print parent classes
        if "parent_classes" in entity_info["reasoning"]:
            print("Parent classes:")
            for parent in entity_info["reasoning"]["parent_classes"]:
                print(f"  - {parent}")
    
    # Get all relationship types
    relationship_types = ontology_access.get_relationship_types()
    print(f"Found {len(relationship_types)} relationship types:")
    for rel_type in relationship_types:
        print(f"  - {rel_type}")


if __name__ == "__main__":
    main()

✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Entity: CHEBI_17790 (CHEBI:17790)
Parent classes:
  - CHEBI_134179
  - CHEBI_15734
  - CHEBI_50584
  - CHEBI_64708
Found 10 relationship types:
  - BFO_0000051
  - RO_0000087
  - has_functional_parent
  - has_major_microspecies_at_pH_7_3
  - has_parent_hydride
  - is_conjugate_acid_of
  - is_conjugate_base_of
  - is_enantiomer_of
  - is_substituent_group_from
  - is_tautomer_of


# basic answer generation

In [10]:
def generate_basic_qa_pairs(chebi_reasoner, entity_id):
    """
    Generate basic QA pairs for a specific entity using real names.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        entity_id: ChEBI ID of the entity
        
    Returns:
        List of QA pair dictionaries
    """
    qa_pairs = []
    
    # Get entity from ontology
    entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
    if not entity_obj or not hasattr(entity_obj, "name") or not entity_obj.name:
        return []
    
    # Get entity name and reasoning info
    entity_name = entity_obj.name
    reasoning = chebi_reasoner._reason_about_entity(entity_id, entity_name)
    
    # Relationship descriptions
    rel_descriptions = {
        "is_a": "{entity_a} is classified as {entity_b}",
        "has_part": "{entity_a} contains {entity_b} as a component",
        "is_conjugate_base_of": "{entity_a} is formed by removing a proton from {entity_b}",
        "is_conjugate_acid_of": "{entity_a} is formed by adding a proton to {entity_b}",
        "is_tautomer_of": "{entity_a} and {entity_b} can interconvert by moving hydrogen atoms",
        "is_enantiomer_of": "{entity_a} is a mirror image of {entity_b}",
        "has_functional_parent": "{entity_a} is derived from {entity_b} by chemical modification",
        "has_parent_hydride": "{entity_a} is derived from {entity_b}",
        "has_role": "{entity_a} functions as {entity_b} in biological or chemical contexts"
    }
    
    # Create a name lookup dictionary for parent classes and related entities
    name_lookup = {}
    
    # Helper function to get real name of an entity
    def get_real_name(item):
        # If it's a string that starts with CHEBI_, try to find its real name
        if isinstance(item, str) and item.startswith("CHEBI_"):
            # Check if we've already looked it up
            if item in name_lookup:
                return name_lookup[item]
                
            # Try to find the entity in the ontology
            for chebi_id, obj in chebi_reasoner.entity_cache.items():
                if obj.name == item:
                    # Get the real name and store it
                    if hasattr(obj, "name") and obj.name:
                        real_name = obj.name
                        name_lookup[item] = real_name
                        return real_name
            
            # If not found, remove the CHEBI_ prefix and use as is
            return item.replace("CHEBI_", "")
        
        # If it's already a real name, return it
        return item
    
    # 1. Generate classification questions
    if "parent_classes" in reasoning and reasoning["parent_classes"]:
        for parent in reasoning["parent_classes"]:
            # Get the real name of the parent class
            parent_name = get_real_name(parent)
            
            # Skip very generic classes
            if parent_name.lower() in ["chemical entity", "molecular entity"]:
                continue
            
            # Yes/No classification question
            question = f"Is {entity_name} a type of {parent_name}?"
            answer = f"Yes, {entity_name} is a type of {parent_name}. This is because {entity_name} has chemical properties and structure consistent with the definition of {parent_name}."
            
            qa_pairs.append({
                "qa_id": f"class_yn_{entity_id}_{hash(parent_name) % 10000}",
                "entity_id": entity_id,
                "entity_name": entity_name,
                "question": question,
                "answer": answer,
                "reasoning_type": "basic",
                "reasoning_subtype": "classification",
                "reasoning_steps": [
                    f"Found that {entity_name} is classified as {parent_name} in the ChEBI ontology",
                    f"This classification is based on chemical structure and properties"
                ]
            })
    
    # 2. Generate property questions
    if "properties" in reasoning and reasoning["properties"]:
        properties = reasoning["properties"]
        
        if "definition" in properties:
            definition = properties["definition"]
            if isinstance(definition, list) and definition:
                definition = definition[0]
            
            if definition and isinstance(definition, str) and len(definition) > 10:
                question = f"What is the definition of {entity_name}?"
                answer = f"{definition}"
                
                qa_pairs.append({
                    "qa_id": f"prop_def_{entity_id}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": "property_definition",
                    "reasoning_steps": [
                        f"Retrieved the formal definition of {entity_name} from the ChEBI ontology",
                        f"This definition characterizes the chemical nature of {entity_name}"
                    ]
                })
    
    # 3. Generate relationship questions
    if "relationships" in reasoning and reasoning["relationships"]:
        for rel_type, targets in reasoning["relationships"].items():
            # Skip is_a relationships as they're covered in classification questions
            if rel_type == "is_a":
                continue
                
            readable_rel = rel_type.replace('_', ' ')
            rel_description = rel_descriptions.get(rel_type, "")
            
            for target in targets:
                # Get the real name of the target entity
                target_name = get_real_name(target)
                
                # Yes/No relationship question
                question = f"Does {entity_name} {readable_rel} {target_name}?"
                
                answer = f"Yes, {entity_name} {readable_rel} {target_name}. "
                if rel_description:
                    answer += f"This means that {rel_description.format(entity_a=entity_name, entity_b=target_name)}."
                
                qa_pairs.append({
                    "qa_id": f"rel_{rel_type}_{entity_id}_{hash(target_name) % 10000}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": f"relationship_{rel_type}",
                    "reasoning_steps": [
                        f"Identified a {readable_rel} relationship between {entity_name} and {target_name}",
                        f"This relationship indicates a specific chemical connection between these entities"
                    ]
                })
    
    return qa_pairs


# Updated dataset generation function that ensures real names
def generate_qa_dataset(chebi_reasoner, output_file="chebi_qa_dataset.json", max_entities=100):
    """Generate a simple QA dataset from ChEBI entities with real names"""
    import json
    import random
    import datetime
    
    print(f"Generating QA dataset with up to {max_entities} entities...")
    
    # Get a sample of entities to process
    all_entity_ids = list(chebi_reasoner.entity_cache.keys())
    random.shuffle(all_entity_ids)
    sample_entity_ids = all_entity_ids[:max_entities]
    
    # Generate QA pairs
    all_qa_pairs = []
    entity_index = {}
    entities_processed = 0
    
    for entity_id in sample_entity_ids:
        qa_pairs = generate_basic_qa_pairs(chebi_reasoner, entity_id)
        
        if qa_pairs:
            # Get entity name
            entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
            if hasattr(entity_obj, "name"):
                entity_name = entity_obj.name
            else:
                entity_name = entity_id
            
            # Add to dataset
            all_qa_pairs.extend(qa_pairs)
            
            # Add to entity index
            entity_index[entity_id] = {
                "name": entity_name,
                "qa_ids": [qa["qa_id"] for qa in qa_pairs]
            }
            
            entities_processed += 1
            
            # Log progress
            if entities_processed % 10 == 0:
                print(f"Processed {entities_processed} entities, generated {len(all_qa_pairs)} QA pairs")
    
    # Create the dataset
    dataset = {
        "metadata": {
            "description": "ChEBI Ontology Question-Answer Dataset",
            "created_on": datetime.datetime.now().isoformat(),
            "num_entities": entities_processed,
            "num_qa_pairs": len(all_qa_pairs)
        },
        "qa_pairs": all_qa_pairs,
        "entity_index": entity_index
    }
    
    # Save to file
    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=2)
    
    print(f"✅ Generated QA dataset with {len(all_qa_pairs)} QA pairs from {entities_processed} entities")
    print(f"✅ Saved to {output_file}")
    
    return dataset

In [11]:
def get_chebi_name(chebi_reasoner, chebi_id):
    """
    Get the name/label of a ChEBI entity.
    
    Args:
        chebi_reasoner: The ChEBI reasoner instance
        chebi_id: ChEBI ID in various formats (CHEBI:12345, CHEBI_12345, etc.)
        
    Returns:
        The entity name or None if not found
    """
    entity = chebi_reasoner._get_entity_by_id(chebi_id)
    
    if entity is None:
        return None
    
    # Try to get the name from various properties
    if hasattr(entity, "label") and entity.label:
        return str(entity.label[0])
    
    # Check if entity has an iri and if we have it in the label cache
    if hasattr(entity, "iri") and hasattr(chebi_reasoner, "label_cache") and entity.iri in chebi_reasoner.label_cache:
        return chebi_reasoner.label_cache[entity.iri]
    
    # Try other properties that might contain the name
    for prop_name in ["title", "name", "chebi_name", "ChEBI_name"]:
        if hasattr(entity, prop_name):
            prop_value = getattr(entity, prop_name)
            if prop_value and len(prop_value) > 0:
                return str(prop_value[0])
    
    # Last resort: use the entity name from IRI
    if hasattr(entity, "name"):
        name = entity.name
        if name and isinstance(name, str) and name.startswith("CHEBI_"):
            name = name[6:]  # Remove the CHEBI_ prefix
        return name
    
    return None


def generate_basic_qa_pairs(chebi_reasoner, entity_id):
    """
    Generate basic QA pairs for a specific entity using real names.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        entity_id: ChEBI ID of the entity
        
    Returns:
        List of QA pair dictionaries
    """
    qa_pairs = []
    
    # Get entity from ontology
    entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
    if not entity_obj:
        return []
    
    # Get entity name using our function
    entity_name = get_chebi_name(chebi_reasoner, entity_id)
    if not entity_name:
        # Fallback if our function doesn't work
        if hasattr(entity_obj, "name") and entity_obj.name:
            entity_name = entity_obj.name
        else:
            return []
    
    # Get reasoning info
    reasoning = chebi_reasoner._reason_about_entity(entity_id, entity_name)
    
    # Relationship descriptions
    rel_descriptions = {
        "is_a": "{entity_a} is classified as {entity_b}",
        "has_part": "{entity_a} contains {entity_b} as a component",
        "is_conjugate_base_of": "{entity_a} is formed by removing a proton from {entity_b}",
        "is_conjugate_acid_of": "{entity_a} is formed by adding a proton to {entity_b}",
        "is_tautomer_of": "{entity_a} and {entity_b} can interconvert by moving hydrogen atoms",
        "is_enantiomer_of": "{entity_a} is a mirror image of {entity_b}",
        "has_functional_parent": "{entity_a} is derived from {entity_b} by chemical modification",
        "has_parent_hydride": "{entity_a} is derived from {entity_b}",
        "has_role": "{entity_a} functions as {entity_b} in biological or chemical contexts"
    }
    
    # Helper function to get real name for any CHEBI ID or string
    def get_name(item):
        # If it looks like a CHEBI ID, convert it
        if isinstance(item, str) and ("CHEBI:" in item or "CHEBI_" in item):
            # Normalize ID format for lookup
            lookup_id = item
            if "CHEBI_" in item:
                # Convert CHEBI_NNNNN to CHEBI:NNNNN format
                lookup_id = item.replace("CHEBI_", "CHEBI:")
            
            # Use our get_chebi_name function to get the real name
            real_name = get_chebi_name(chebi_reasoner, lookup_id)
            return real_name if real_name else item
        
        # Otherwise return as is
        return item
    
    # 1. Generate classification questions
    if "parent_classes" in reasoning and reasoning["parent_classes"]:
        for parent in reasoning["parent_classes"]:
            # Get the real name of the parent class
            parent_name = get_name(parent)
            
            # Skip very generic classes
            if parent_name.lower() in ["chemical entity", "molecular entity"]:
                continue
            
            # Yes/No classification question
            question = f"Is {entity_name} a type of {parent_name}?"
            answer = f"Yes, {entity_name} is a type of {parent_name}. This is because {entity_name} has chemical properties and structure consistent with the definition of {parent_name}."
            
            qa_pairs.append({
                "qa_id": f"class_yn_{entity_id}_{hash(parent_name) % 10000}",
                "entity_id": entity_id,
                "entity_name": entity_name,
                "question": question,
                "answer": answer,
                "reasoning_type": "basic",
                "reasoning_subtype": "classification",
                "reasoning_steps": [
                    f"Found that {entity_name} is classified as {parent_name} in the ChEBI ontology",
                    f"This classification is based on chemical structure and properties"
                ]
            })
    
    # 2. Generate property questions
    if "properties" in reasoning and reasoning["properties"]:
        properties = reasoning["properties"]
        
        if "definition" in properties:
            definition = properties["definition"]
            if isinstance(definition, list) and definition:
                definition = definition[0]
            
            if definition and isinstance(definition, str) and len(definition) > 10:
                question = f"What is the definition of {entity_name}?"
                answer = f"{definition}"
                
                qa_pairs.append({
                    "qa_id": f"prop_def_{entity_id}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": "property_definition",
                    "reasoning_steps": [
                        f"Retrieved the formal definition of {entity_name} from the ChEBI ontology",
                        f"This definition characterizes the chemical nature of {entity_name}"
                    ]
                })
    
    # 3. Generate relationship questions
    if "relationships" in reasoning and reasoning["relationships"]:
        for rel_type, targets in reasoning["relationships"].items():
            # Skip is_a relationships as they're covered in classification questions
            if rel_type == "is_a":
                continue
                
            readable_rel = rel_type.replace('_', ' ')
            rel_description = rel_descriptions.get(rel_type, "")
            
            for target in targets:
                # Get the real name of the target entity
                target_name = get_name(target)
                
                # Yes/No relationship question
                question = f"Does {entity_name} {readable_rel} {target_name}?"
                
                answer = f"Yes, {entity_name} {readable_rel} {target_name}. "
                if rel_description:
                    answer += f"This means that {rel_description.format(entity_a=entity_name, entity_b=target_name)}."
                
                qa_pairs.append({
                    "qa_id": f"rel_{rel_type}_{entity_id}_{hash(target_name) % 10000}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": f"relationship_{rel_type}",
                    "reasoning_steps": [
                        f"Identified a {readable_rel} relationship between {entity_name} and {target_name}",
                        f"This relationship indicates a specific chemical connection between these entities"
                    ]
                })
    
    return qa_pairs


def generate_qa_dataset(chebi_reasoner, output_file="chebi_qa_dataset.json", max_entities=100):
    """Generate a simple QA dataset from ChEBI entities with real names"""
    import json
    import random
    import datetime
    
    print(f"Generating QA dataset with up to {max_entities} entities...")
    
    # Get a sample of entities to process
    all_entity_ids = list(chebi_reasoner.entity_cache.keys())
    random.shuffle(all_entity_ids)
    sample_entity_ids = all_entity_ids[:max_entities]
    
    # Generate QA pairs
    all_qa_pairs = []
    entity_index = {}
    entities_processed = 0
    
    for entity_id in sample_entity_ids:
        qa_pairs = generate_basic_qa_pairs(chebi_reasoner, entity_id)
        
        if qa_pairs:
            # Get entity name using our function
            entity_name = get_chebi_name(chebi_reasoner, entity_id)
            if not entity_name:
                entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
                if hasattr(entity_obj, "name"):
                    entity_name = entity_obj.name
                else:
                    entity_name = entity_id
            
            # Add to dataset
            all_qa_pairs.extend(qa_pairs)
            
            # Add to entity index
            entity_index[entity_id] = {
                "name": entity_name,
                "qa_ids": [qa["qa_id"] for qa in qa_pairs]
            }
            
            entities_processed += 1
            
            # Log progress
            if entities_processed % 10 == 0:
                print(f"Processed {entities_processed} entities, generated {len(all_qa_pairs)} QA pairs")
    
    # Create the dataset
    dataset = {
        "metadata": {
            "description": "ChEBI Ontology Question-Answer Dataset",
            "created_on": datetime.datetime.now().isoformat(),
            "num_entities": entities_processed,
            "num_qa_pairs": len(all_qa_pairs)
        },
        "qa_pairs": all_qa_pairs,
        "entity_index": entity_index
    }
    
    # Save to file
    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=2)
    
    print(f"✅ Generated QA dataset with {len(all_qa_pairs)} QA pairs from {entities_processed} entities")
    print(f"✅ Saved to {output_file}")
    
    return dataset

In [12]:
"""
Test script for the ChEBI QA Generator.
This script demonstrates how to generate question-answer pairs
from the ChEBI ontology using our custom generator.
"""

import sys
import os
import json
from pprint import pprint

# Import the ChEBIReasoner class
# Adjust the import path as needed for your environment
from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
chebi_owl_path = '/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl'

# Import the QA generator function
#from chebi_simple_qa_generator import generate_basic_qa_pairs, generate_qa_dataset

def test_single_entity(chebi_owl_path, entity_id="CHEBI:17790"):
    """Test QA generation for a single entity (default is methanol)"""
    print(f"Testing QA generation for entity {entity_id}...")
    
    # Initialize the ChEBI reasoner
    chebi_reasoner = ChEBIReasoner(chebi_owl_path)
    
    # Get entity from ontology to display name
    entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
    if entity_obj and hasattr(entity_obj, "name"):
        print(f"Entity name: {entity_obj.name}")
    
    # Generate QA pairs
    qa_pairs = generate_basic_qa_pairs(chebi_reasoner, entity_id)
    
    # Display the results
    print(f"Generated {len(qa_pairs)} QA pairs:")
    for i, qa in enumerate(qa_pairs, 1):
        print(f"\nQA Pair #{i}:")
        print(f"Question: {qa['question']}")
        print(f"Answer: {qa['answer']}")
        print(f"Type: {qa['reasoning_type']}/{qa['reasoning_subtype']}")
        print("Reasoning steps:")
        for step in qa['reasoning_steps']:
            print(f"  - {step}")

def test_small_dataset(chebi_owl_path, output_file="test_chebi_qa_dataset.json", num_entities=10):
    """Test generation of a small QA dataset"""
    print(f"Testing dataset generation with {num_entities} entities...")
    
    # Initialize the ChEBI reasoner
    chebi_reasoner = ChEBIReasoner(chebi_owl_path)
    
    # Generate and save a small dataset
    dataset = generate_qa_dataset(chebi_reasoner, output_file, max_entities=num_entities)
    
    # Display summary statistics
    print("\nDataset Statistics:")
    print(f"Number of entities processed: {dataset['metadata']['num_entities']}")
    print(f"Number of QA pairs generated: {dataset['metadata']['num_qa_pairs']}")
    
    # Display a sample QA pair
    if dataset['qa_pairs']:
        print("\nSample QA pair:")
        sample_qa = dataset['qa_pairs'][0]
        print(f"Question: {sample_qa['question']}")
        print(f"Answer: {sample_qa['answer']}")
        print(f"Type: {sample_qa['reasoning_type']}/{sample_qa['reasoning_subtype']}")
    
    print(f"\nDataset saved to {output_file}")

def test_specific_entities(chebi_owl_path, entity_ids=None):
    """Test QA generation for specific entities of interest"""
    if entity_ids is None:
        # Default list of interesting chemical entities
        entity_ids = [
            "CHEBI:17790",  # Methanol
            "CHEBI:15377",  # Water
            "CHEBI:15354",  # Aspirin
            "CHEBI:16526",  # Carbon dioxide
            "CHEBI:15428"   # Ethanol
        ]
    
    print(f"Testing QA generation for {len(entity_ids)} specific entities...")
    
    # Initialize the ChEBI reasoner
    chebi_reasoner = ChEBIReasoner(chebi_owl_path)
    
    # Process each entity
    all_qa_pairs = []
    
    for entity_id in entity_ids:
        entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
        if entity_obj and hasattr(entity_obj, "name"):
            entity_name = entity_obj.name
            print(f"\nProcessing: {entity_name} ({entity_id})")
            
            qa_pairs = generate_basic_qa_pairs(chebi_reasoner, entity_id)
            all_qa_pairs.extend(qa_pairs)
            
            print(f"Generated {len(qa_pairs)} QA pairs")
            
            # Display one random QA pair as example
            if qa_pairs:
                import random
                sample_qa = random.choice(qa_pairs)
                print("Sample QA pair:")
                print(f"Q: {sample_qa['question']}")
                print(f"A: {sample_qa['answer']}")
    
    print(f"\nTotal QA pairs generated: {len(all_qa_pairs)}")

if __name__ == "__main__":
    # Path to ChEBI OWL file
    from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
    chebi_owl_path = '/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl'
    
    # Initialize the reasoner with your ontology file
    chebi_reasoner = ChEBIReasoner("/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl")
    
    # Uncomment the test you want to run
    
    # Test for a single entity
    test_single_entity(chebi_owl_path)
    
    # Test for a small dataset
    # test_small_dataset(chebi_owl_path)
    
    # Test for specific entities of interest
    # test_specific_entities(chebi_owl_path)

✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Testing QA generation for entity CHEBI:17790...
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Entity name: CHEBI_17790
Generated 11 QA pairs:

QA Pair #1:
Question: Is methanol a type of volatile organic compound?
Answer: Yes, methanol is a type of volatile organic compound. This is because methanol has chemical properties and structure consistent with the definition of volatile organic compound.
Type: basic/classification
Reasoning steps:
  - Found that methanol is classified as volatile organic compound in the ChEBI ontology
  - This classification is based on chemical structure and properties

QA Pair #2:
Question: Is methanol a type of primary alcohol?
Answer: Yes, methanol is a type of primary alcohol. This is because methanol has chemical properties and structure consistent with the definition of primary alcoho

# comparitive questions 

In [13]:
def generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id):
    """
    Generate comparative QA pairs that compare two chemical entities.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        entity1_id: ChEBI ID of the first entity
        entity2_id: ChEBI ID of the second entity
        
    Returns:
        List of comparative QA pair dictionaries
    """
    qa_pairs = []
    
    # Get entity names
    entity1_name = get_chebi_name(chebi_reasoner, entity1_id)
    entity2_name = get_chebi_name(chebi_reasoner, entity2_id)
    
    if not entity1_name or not entity2_name:
        return []
    
    # Get reasoning info for both entities
    reasoning1 = chebi_reasoner._reason_about_entity(entity1_id, entity1_name)
    reasoning2 = chebi_reasoner._reason_about_entity(entity2_id, entity2_name)
    
    # Helper function to get name
    def get_name(item):
        if isinstance(item, str) and ("CHEBI:" in item or "CHEBI_" in item):
            lookup_id = item
            if "CHEBI_" in item:
                lookup_id = item.replace("CHEBI_", "CHEBI:")
            name = get_chebi_name(chebi_reasoner, lookup_id)
            return name if name else item
        return item
    
    # 1. Compare classifications
    parent_classes1 = set()
    parent_classes2 = set()
    
    if "parent_classes" in reasoning1 and reasoning1["parent_classes"]:
        for parent in reasoning1["parent_classes"]:
            parent_name = get_name(parent)
            if parent_name.lower() not in ["chemical entity", "molecular entity"]:
                parent_classes1.add(parent_name)
    
    if "parent_classes" in reasoning2 and reasoning2["parent_classes"]:
        for parent in reasoning2["parent_classes"]:
            parent_name = get_name(parent)
            if parent_name.lower() not in ["chemical entity", "molecular entity"]:
                parent_classes2.add(parent_name)
    
    # Find common and unique classifications
    common_classes = parent_classes1.intersection(parent_classes2)
    unique_classes1 = parent_classes1 - parent_classes2
    unique_classes2 = parent_classes2 - parent_classes1
    
    if common_classes or (unique_classes1 and unique_classes2):
        # Create comparison question
        question = f"How do {entity1_name} and {entity2_name} compare in terms of their chemical classification?"
        
        answer_parts = []
        if common_classes:
            common_str = ", ".join(list(common_classes)[:3])
            answer_parts.append(f"Both {entity1_name} and {entity2_name} are classified as {common_str}.")
        
        if unique_classes1:
            unique_str1 = ", ".join(list(unique_classes1)[:3])
            answer_parts.append(f"{entity1_name} is additionally classified as {unique_str1}, which {entity2_name} is not.")
        
        if unique_classes2:
            unique_str2 = ", ".join(list(unique_classes2)[:3])
            answer_parts.append(f"{entity2_name} is additionally classified as {unique_str2}, which {entity1_name} is not.")
        
        answer = " ".join(answer_parts)
        
        qa_pairs.append({
            "qa_id": f"comparative_classification_{entity1_id}_{entity2_id}",
            "entity1_id": entity1_id,
            "entity1_name": entity1_name,
            "entity2_id": entity2_id,
            "entity2_name": entity2_name,
            "question": question,
            "answer": answer,
            "reasoning_type": "comparative",
            "reasoning_subtype": "classification_comparison",
            "reasoning_steps": [
                f"Retrieved classifications for both {entity1_name} and {entity2_name}",
                f"Compared their chemical classifications to identify similarities and differences"
            ]
        })
    
    # 2. Compare structural components (has_part relationships)
    parts1 = set()
    parts2 = set()
    
    # Get parts for both entities
    if "relationships" in reasoning1 and "has_part" in reasoning1["relationships"]:
        for part in reasoning1["relationships"]["has_part"]:
            parts1.add(get_name(part))
    
    if "relationships" in reasoning2 and "has_part" in reasoning2["relationships"]:
        for part in reasoning2["relationships"]["has_part"]:
            parts2.add(get_name(part))
    
    if parts1 or parts2:
        # Create question about structural comparison
        question = f"What structural components differ between {entity1_name} and {entity2_name}?"
        
        # Find common and unique parts
        common_parts = parts1.intersection(parts2)
        unique_parts1 = parts1 - parts2
        unique_parts2 = parts2 - parts1
        
        answer_parts = []
        
        if common_parts:
            common_str = ", ".join(list(common_parts)[:3])
            answer_parts.append(f"Both compounds contain {common_str}.")
        
        if unique_parts1:
            unique_str1 = ", ".join(list(unique_parts1)[:3])
            answer_parts.append(f"{entity1_name} uniquely contains {unique_str1}.")
        
        if unique_parts2:
            unique_str2 = ", ".join(list(unique_parts2)[:3])
            answer_parts.append(f"{entity2_name} uniquely contains {unique_str2}.")
        
        if not answer_parts:
            answer_parts = [f"Insufficient structural information is available to compare these compounds."]
        
        answer = " ".join(answer_parts)
        
        qa_pairs.append({
            "qa_id": f"comparative_structural_{entity1_id}_{entity2_id}",
            "entity1_id": entity1_id,
            "entity1_name": entity1_name,
            "entity2_id": entity2_id,
            "entity2_name": entity2_name,
            "question": question,
            "answer": answer,
            "reasoning_type": "comparative",
            "reasoning_subtype": "structural_comparison",
            "reasoning_steps": [
                f"Identified structural components of both chemicals",
                f"Compared their composition to determine similarities and differences"
            ]
        })
    
    return qa_pairs


def find_related_entity_pairs(chebi_reasoner, max_pairs=50):
    """
    Find pairs of related entities that would be interesting to compare.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        max_pairs: Maximum number of entity pairs to return
        
    Returns:
        List of (entity1_id, entity2_id) tuples
    """
    entity_pairs = []
    processed_pairs = set()
    
    # Strategy: Find entities with common parent classes (siblings)
    class_to_entities = {}
    
    # Sample a subset of entities for efficiency
    import random
    all_entity_ids = list(chebi_reasoner.entity_cache.keys())
    random.shuffle(all_entity_ids)
    sample_entities = all_entity_ids[:500]  # Limit to 500 entities for efficiency
    
    # Build a map of classes to entities
    for entity_id in sample_entities:
        entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
        if not entity_obj or not hasattr(entity_obj, "is_a"):
            continue
        
        parents = entity_obj.is_a
        if not isinstance(parents, list):
            parents = [parents]
        
        for parent in parents:
            if not hasattr(parent, "name") or not parent.name:
                continue
                
            parent_name = parent.name
            
            if parent_name not in class_to_entities:
                class_to_entities[parent_name] = []
            
            class_to_entities[parent_name].append(entity_id)
    
    # Find siblings (entities with common parent classes)
    for parent_name, entities in class_to_entities.items():
        if len(entities) < 2:
            continue
            
        # Skip very general classes that would yield too many siblings
        if parent_name.lower() in ["chemical entity", "molecular entity"]:
            continue
        
        # Take up to 3 pairs from each class
        import itertools
        for entity1_id, entity2_id in list(itertools.combinations(entities, 2))[:3]:
            pair_key = tuple(sorted([entity1_id, entity2_id]))
            
            if pair_key in processed_pairs:
                continue
                
            processed_pairs.add(pair_key)
            entity_pairs.append((entity1_id, entity2_id))
            
            if len(entity_pairs) >= max_pairs:
                return entity_pairs
    
    return entity_pairs


def add_comparative_qa_to_dataset(chebi_reasoner, num_entity_pairs=25):
    """
    Find related entity pairs and add comparative QA pairs to a dataset.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        num_entity_pairs: Number of entity pairs to use
        
    Returns:
        List of comparative QA pairs
    """
    print(f"Generating comparative QA pairs for {num_entity_pairs} entity pairs...")
    
    # Find related entity pairs
    entity_pairs = find_related_entity_pairs(chebi_reasoner, max_pairs=num_entity_pairs)
    
    # Generate QA pairs for each entity pair
    all_qa_pairs = []
    
    for entity1_id, entity2_id in entity_pairs:
        entity1_name = get_chebi_name(chebi_reasoner, entity1_id)
        entity2_name = get_chebi_name(chebi_reasoner, entity2_id)
        
        if entity1_name and entity2_name:
            print(f"Comparing {entity1_name} and {entity2_name}...")
            
            qa_pairs = generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id)
            all_qa_pairs.extend(qa_pairs)
            
            print(f"Generated {len(qa_pairs)} comparative QA pairs")
    
    print(f"Total comparative QA pairs generated: {len(all_qa_pairs)}")
    return all_qa_pairs

In [14]:
"""
Test script for the ChEBI Comparative QA Generator.
This script demonstrates generating comparative question-answer pairs
between related chemical entities in the ChEBI ontology.
"""

import sys
import os
import json
from pprint import pprint

# Import the ChEBIReasoner class and QA generators
# Adjust the import path as needed for your environment
from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
chebi_owl_path = '/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl'
#from chebi_qa_generator import get_chebi_name  # Import the name resolution function
#from simple_comparative_qa import generate_comparative_qa_pairs, find_related_entity_pairs


def test_specific_pair(chebi_reasoner, entity1_id="CHEBI:17790", entity2_id="CHEBI:15377"):
    """
    Test comparative QA generation for a specific pair of entities.
    Default: Methanol (CHEBI:17790) and Water (CHEBI:15377)
    """
    print(f"Testing comparative QA generation for entity pair: {entity1_id} and {entity2_id}...")
    
    # Get entity names
    entity1_name = get_chebi_name(chebi_reasoner, entity1_id) or entity1_id
    entity2_name = get_chebi_name(chebi_reasoner, entity2_id) or entity2_id
    
    print(f"Comparing: {entity1_name} and {entity2_name}")
    
    # Generate comparative QA pairs
    qa_pairs = generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id)
    
    # Display the results
    print(f"Generated {len(qa_pairs)} comparative QA pairs:")
    for i, qa in enumerate(qa_pairs, 1):
        print(f"\nComparative QA Pair #{i}:")
        print(f"Question: {qa['question']}")
        print(f"Answer: {qa['answer']}")
        print(f"Type: {qa['reasoning_type']}/{qa['reasoning_subtype']}")
        print("Reasoning steps:")
        for step in qa['reasoning_steps']:
            print(f"  - {step}")


def test_find_related_pairs(chebi_reasoner, num_pairs=5):
    """
    Test finding related entity pairs for comparison.
    """
    print(f"Finding {num_pairs} related entity pairs for comparison...")
    
    # Find related entity pairs
    entity_pairs = find_related_entity_pairs(chebi_reasoner, max_pairs=num_pairs)
    
    # Display the results
    print(f"Found {len(entity_pairs)} related entity pairs:")
    for i, (entity1_id, entity2_id) in enumerate(entity_pairs, 1):
        entity1_name = get_chebi_name(chebi_reasoner, entity1_id) or entity1_id
        entity2_name = get_chebi_name(chebi_reasoner, entity2_id) or entity2_id
        
        print(f"{i}. {entity1_name} ({entity1_id}) and {entity2_name} ({entity2_id})")


def test_multiple_pairs(chebi_reasoner, num_pairs=3):
    """
    Test comparative QA generation for multiple pairs of entities.
    """
    print(f"Testing comparative QA generation for {num_pairs} entity pairs...")
    
    # Find related entity pairs
    entity_pairs = find_related_entity_pairs(chebi_reasoner, max_pairs=num_pairs)
    
    total_qa_pairs = 0
    
    # Generate and display QA pairs for each entity pair
    for entity1_id, entity2_id in entity_pairs:
        entity1_name = get_chebi_name(chebi_reasoner, entity1_id) or entity1_id
        entity2_name = get_chebi_name(chebi_reasoner, entity2_id) or entity2_id
        
        print(f"\n=== Comparing: {entity1_name} and {entity2_name} ===")
        
        # Generate comparative QA pairs
        qa_pairs = generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id)
        total_qa_pairs += len(qa_pairs)
        
        # Display sample QA pair
        if qa_pairs:
            print(f"Generated {len(qa_pairs)} comparative QA pairs. Sample:")
            sample_qa = qa_pairs[0]
            print(f"Q: {sample_qa['question']}")
            print(f"A: {sample_qa['answer']}")
        else:
            print("No comparative QA pairs generated for this pair.")
    
    print(f"\nTotal comparative QA pairs generated: {total_qa_pairs}")


def test_specific_chemical_types(chebi_reasoner):
    """
    Test comparative QA generation for specific types of chemicals.
    """
    # Common chemical pairs that should yield interesting comparisons
    chemical_pairs = [
        ("CHEBI:15377", "CHEBI:15379"),  # Water vs Heavy Water
        ("CHEBI:17790", "CHEBI:15428"),  # Methanol vs Ethanol  
        ("CHEBI:15365", "CHEBI:15366"),  # Acetaminophen vs Ibuprofen
        ("CHEBI:16526", "CHEBI:17245"),  # Carbon dioxide vs Carbon monoxide
        ("CHEBI:16933", "CHEBI:27225")   # Sodium chloride vs Potassium chloride
    ]
    
    print("Testing comparative QA generation for specific chemical pairs:")
    
    for entity1_id, entity2_id in chemical_pairs:
        entity1_name = get_chebi_name(chebi_reasoner, entity1_id) or entity1_id
        entity2_name = get_chebi_name(chebi_reasoner, entity2_id) or entity2_id
        
        print(f"\n=== Comparing: {entity1_name} and {entity2_name} ===")
        
        # Generate comparative QA pairs
        qa_pairs = generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id)
        
        # Display number of QA pairs
        print(f"Generated {len(qa_pairs)} comparative QA pairs")
        
        # Display one sample QA pair if available
        if qa_pairs:
            sample_qa = qa_pairs[0]
            print("Sample comparison:")
            print(f"Q: {sample_qa['question']}")
            print(f"A: {sample_qa['answer']}")


def save_comparative_dataset(chebi_reasoner, output_file="comparative_qa_dataset.json", num_pairs=25):
    """
    Generate and save a dataset of comparative QA pairs.
    """
    print(f"Generating comparative QA dataset with {num_pairs} entity pairs...")
    
    # Find related entity pairs
    entity_pairs = find_related_entity_pairs(chebi_reasoner, max_pairs=num_pairs)
    
    # Generate QA pairs
    all_qa_pairs = []
    entity_index = {}
    
    for entity1_id, entity2_id in entity_pairs:
        entity1_name = get_chebi_name(chebi_reasoner, entity1_id) or entity1_id
        entity2_name = get_chebi_name(chebi_reasoner, entity2_id) or entity2_id
        
        print(f"Comparing {entity1_name} and {entity2_name}...")
        
        # Generate comparative QA pairs
        qa_pairs = generate_comparative_qa_pairs(chebi_reasoner, entity1_id, entity2_id)
        all_qa_pairs.extend(qa_pairs)
        
        # Update entity index
        if entity1_id not in entity_index:
            entity_index[entity1_id] = {"name": entity1_name, "qa_ids": []}
        if entity2_id not in entity_index:
            entity_index[entity2_id] = {"name": entity2_name, "qa_ids": []}
        
        # Add QA IDs to entity index
        for qa in qa_pairs:
            entity_index[entity1_id]["qa_ids"].append(qa["qa_id"])
            entity_index[entity2_id]["qa_ids"].append(qa["qa_id"])
        
        print(f"Generated {len(qa_pairs)} comparative QA pairs")
    
    # Create dataset
    import datetime
    dataset = {
        "metadata": {
            "description": "ChEBI Comparative Question-Answer Dataset",
            "created_on": datetime.datetime.now().isoformat(),
            "num_entity_pairs": len(entity_pairs),
            "num_qa_pairs": len(all_qa_pairs)
        },
        "qa_pairs": all_qa_pairs,
        "entity_index": entity_index
    }
    
    # Save to file
    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=2)
    
    print(f"✅ Saved comparative QA dataset with {len(all_qa_pairs)} QA pairs to {output_file}")
    return dataset


if __name__ == "__main__":
    # Path to ChEBI OWL file
    chebi_owl_path = '/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl' # Update this to your actual path
    
    # Initialize the ChEBI reasoner
    chebi_reasoner = ChEBIReasoner(chebi_owl_path)
    
    # Uncomment the test you want to run
    
    # Test for a specific pair of chemicals
    test_specific_pair(chebi_reasoner, "CHEBI:17790", "CHEBI:15428")  # Methanol vs Ethanol
    
    # Test finding related entity pairs
    test_find_related_pairs(chebi_reasoner, num_pairs=5)
    
    # Test for multiple pairs
    test_multiple_pairs(chebi_reasoner, num_pairs=3)
    
    # Test for specific chemical types
    test_specific_chemical_types(chebi_reasoner)
    
    # Generate and save a comparative QA dataset
    save_comparative_dataset(chebi_reasoner, num_pairs=100)

✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Testing comparative QA generation for entity pair: CHEBI:17790 and CHEBI:15428...
Comparing: methanol and glycine
Generated 1 comparative QA pairs:

Comparative QA Pair #1:
Question: How do methanol and glycine compare in terms of their chemical classification?
Answer: methanol is additionally classified as alkyl alcohol, one-carbon compound, primary alcohol, which glycine is not. glycine is additionally classified as alpha-amino acid, serine family amino acid, proteinogenic amino acid, which methanol is not.
Type: comparative/classification_comparison
Reasoning steps:
  - Retrieved classifications for both methanol and glycine
  - Compared their chemical classifications to identify similarities and differences
Finding 5 related entity pairs for comparison...
Found 5 related entity pairs:
1. O(3)-(beta-D-xylosyl)-L-serine residue (CHEBI:132085) and L-tyrosine residue (CHEBI:4685

In [22]:
def get_real_names_mapping(chebi_reasoner):
    """
    Create a mapping from CHEBI IDs and CHEBI_* format strings to real names.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        
    Returns:
        Dictionary mapping entity IDs to their real names
    """
    name_mapping = {}
    
    # Build mapping for all entities in the cache
    for chebi_id, entity_obj in chebi_reasoner.entity_cache.items():
        if hasattr(entity_obj, "name") and entity_obj.name:
            # Map from ChEBI ID to name
            name_mapping[chebi_id] = entity_obj.name
            
            # Also map from the CHEBI_* format (which appears in reasoning)
            if "CHEBI:" in chebi_id:
                chebi_underscore = chebi_id.replace("CHEBI:", "CHEBI_")
                name_mapping[chebi_underscore] = entity_obj.name
    
    return name_mapping


def generate_qa_pairs_with_names(chebi_reasoner, entity_id):
    """
    Generate QA pairs using real entity names instead of IDs.
    
    Args:
        chebi_reasoner: Initialized ChEBIReasoner instance
        entity_id: ChEBI ID of the entity
        
    Returns:
        List of QA pair dictionaries
    """
    # First build the name mapping
    name_mapping = get_real_names_mapping(chebi_reasoner)
    
    # Get entity info
    entity_obj = chebi_reasoner._get_entity_by_id(entity_id)
    if not entity_obj or not hasattr(entity_obj, "name") or not entity_obj.name:
        return []
    
    entity_name = entity_obj.name
    reasoning = chebi_reasoner._reason_about_entity(entity_id, entity_name)
    
    # Function to get real name
    def get_name(id_or_name):
        if id_or_name in name_mapping:
            return name_mapping[id_or_name]
        return id_or_name
    
    # Relationship descriptions
    rel_descriptions = {
        "is_a": "{entity_a} is classified as {entity_b}",
        "has_part": "{entity_a} contains {entity_b} as a component",
        "is_conjugate_base_of": "{entity_a} is formed by removing a proton from {entity_b}",
        "is_conjugate_acid_of": "{entity_a} is formed by adding a proton to {entity_b}",
        "is_tautomer_of": "{entity_a} and {entity_b} can interconvert by moving hydrogen atoms",
        "is_enantiomer_of": "{entity_a} is a mirror image of {entity_b}",
        "has_functional_parent": "{entity_a} is derived from {entity_b} by chemical modification",
        "has_parent_hydride": "{entity_a} is derived from {entity_b}",
        "has_role": "{entity_a} functions as {entity_b} in biological or chemical contexts"
    }
    
    qa_pairs = []
    
    # 1. Classification questions
    if "parent_classes" in reasoning and reasoning["parent_classes"]:
        for parent in reasoning["parent_classes"]:
            parent_name = get_name(parent)
            
            # Skip generic classes
            if parent_name.lower() in ["chemical entity", "molecular entity"]:
                continue
            
            question = f"Is {entity_name} a type of {parent_name}?"
            answer = f"Yes, {entity_name} is a type of {parent_name}. This is because {entity_name} has chemical properties and structure consistent with the definition of {parent_name}."
            
            qa_pairs.append({
                "qa_id": f"class_yn_{entity_id}_{hash(parent_name) % 10000}",
                "entity_id": entity_id,
                "entity_name": entity_name,
                "question": question,
                "answer": answer,
                "reasoning_type": "basic",
                "reasoning_subtype": "classification",
                "reasoning_steps": [
                    f"Found that {entity_name} is classified as {parent_name} in the ChEBI ontology",
                    f"This classification is based on chemical structure and properties"
                ]
            })
    
    # 2. Relationship questions
    if "relationships" in reasoning and reasoning["relationships"]:
        for rel_type, targets in reasoning["relationships"].items():
            if rel_type == "is_a":
                continue
                
            readable_rel = rel_type.replace('_', ' ')
            
            for target in targets:
                target_name = get_name(target)
                
                question = f"Does {entity_name} {readable_rel} {target_name}?"
                answer = f"Yes, {entity_name} {readable_rel} {target_name}. "
                
                if rel_type in rel_descriptions:
                    answer += f"This means that {rel_descriptions[rel_type].format(entity_a=entity_name, entity_b=target_name)}."
                
                qa_pairs.append({
                    "qa_id": f"rel_{rel_type}_{entity_id}_{hash(target_name) % 10000}",
                    "entity_id": entity_id,
                    "entity_name": entity_name,
                    "question": question,
                    "answer": answer,
                    "reasoning_type": "basic",
                    "reasoning_subtype": f"relationship_{rel_type}",
                    "reasoning_steps": [
                        f"Identified a {readable_rel} relationship between {entity_name} and {target_name}",
                        f"This relationship indicates a specific chemical connection between these entities"
                    ]
                })
    
    return qa_pairs


# Example usage for testing
def test_with_name_mapping(chebi_reasoner, entity_id="CHEBI:17790"):
    """Test QA generation with proper name mapping"""
    qa_pairs = generate_qa_pairs_with_names(chebi_reasoner, entity_id)
    
    print(f"Generated {len(qa_pairs)} QA pairs with real names:")
    for i, qa in enumerate(qa_pairs, 1):
        print(f"\nQA Pair #{i}:")
        print(f"Question: {qa['question']}")
        print(f"Answer: {qa['answer']}")
        print(f"Type: {qa['reasoning_type']}/{qa['reasoning_subtype']}")
        print("Reasoning steps:")
        for step in qa['reasoning_steps']:
            print(f"  - {step}")

In [23]:
test_with_name_mapping(ChEBIReasoner, entity_id="CHEBI:17790")

AttributeError: type object 'ChEBIReasoner' has no attribute 'entity_cache'

In [15]:
"""
ChEBI QA Dataset Analysis and Visualization - Streamlined Version
"""
import json
import matplotlib.pyplot as plt
import os
from collections import Counter
import networkx as nx
import numpy as np

def analyze_chebi_qa_dataset(dataset_path, output_dir=None):
    """Analyze ChEBI QA dataset and generate visualizations"""
    # Create output directory if needed
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Load dataset
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)
    
    # Extract key statistics
    entity_count = len(dataset['entity_index'])
    qa_count = len(dataset['qa_pairs'])
    
    # Extract reasoning types
    reasoning_types = Counter([qa.get('reasoning_type', 'unknown') for qa in dataset['qa_pairs']])
    reasoning_subtypes = Counter([qa.get('reasoning_subtype', 'unknown') for qa in dataset['qa_pairs']])
    
    # Calculate question/answer lengths
    q_lengths = [len(qa['question'].split()) for qa in dataset['qa_pairs']]
    a_lengths = [len(qa['answer'].split()) for qa in dataset['qa_pairs']]
    
    # Count QA pairs per entity
    entity_qa_counts = [len(info['qa_ids']) for info in dataset['entity_index'].values()]
    
    # Generate plots
    generate_plots(
        reasoning_types, 
        reasoning_subtypes, 
        q_lengths, 
        a_lengths, 
        entity_qa_counts,
        dataset,
        output_dir
    )
    
    # Generate summary
    summary_stats = {
        "total_entities": entity_count,
        "total_qa_pairs": qa_count,
        "avg_qa_per_entity": qa_count / entity_count,
        "avg_question_length": np.mean(q_lengths),
        "avg_answer_length": np.mean(a_lengths),
        "reasoning_types": reasoning_types,
        "reasoning_subtypes": reasoning_subtypes
    }
    
    # Generate summary text
    summary_text = generate_summary(summary_stats)
    
    # Save summary
    if output_dir:
        with open(os.path.join(output_dir, "dataset_summary.md"), "w") as f:
            f.write(summary_text)
    
    print(f"✅ Analysis complete! Results saved to {output_dir}")
    return summary_stats

def generate_plots(reasoning_types, reasoning_subtypes, q_lengths, a_lengths, entity_qa_counts, dataset, output_dir=None):
    """Generate all visualization plots"""
    
    # Plot 1: Reasoning Types Distribution
    plt.figure(figsize=(10, 6))
    types = list(reasoning_types.keys())
    counts = list(reasoning_types.values())
    # Sort by count
    sorted_indices = np.argsort(counts)[::-1]
    types = [types[i] for i in sorted_indices]
    counts = [counts[i] for i in sorted_indices]
    
    plt.barh(types, counts, color='skyblue')
    plt.xlabel('Number of QA Pairs')
    plt.ylabel('Reasoning Type')
    plt.title('Distribution of QA Pairs by Reasoning Type')
    
    for i, v in enumerate(counts):
        plt.text(v + 0.5, i, str(v), color='blue', fontweight='bold')
    
    plt.tight_layout()
    if output_dir:
        plt.savefig(os.path.join(output_dir, 'reasoning_types.png'), dpi=300)
        plt.close()
    
    # Plot 2: Top Reasoning Subtypes
    plt.figure(figsize=(12, 8))
    top_subtypes = dict(sorted(reasoning_subtypes.items(), key=lambda x: x[1], reverse=True)[:10])
    types = list(top_subtypes.keys())
    counts = list(top_subtypes.values())
    
    plt.barh(types, counts, color='lightgreen')
    plt.xlabel('Number of QA Pairs')
    plt.ylabel('Reasoning Subtype')
    plt.title('Top 10 Reasoning Subtypes')
    
    for i, v in enumerate(counts):
        plt.text(v + 0.5, i, str(v), color='darkgreen', fontweight='bold')
    
    plt.tight_layout()
    if output_dir:
        plt.savefig(os.path.join(output_dir, 'reasoning_subtypes.png'), dpi=300)
        plt.close()
    
    # Plot 3: Question & Answer Lengths
    plt.figure(figsize=(10, 6))
    plt.boxplot([q_lengths, a_lengths], labels=['Question Length', 'Answer Length'], 
                patch_artist=True, boxprops=dict(facecolor='lightblue'))
    plt.ylabel('Word Count')
    plt.title('Question and Answer Lengths')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    if output_dir:
        plt.savefig(os.path.join(output_dir, 'qa_lengths.png'), dpi=300)
        plt.close()
    
    # Plot 4: QA Pairs per Entity
    plt.figure(figsize=(10, 6))
    bins = min(20, max(entity_qa_counts) - min(entity_qa_counts) + 1)
    plt.hist(entity_qa_counts, bins=bins, color='salmon', edgecolor='black', alpha=0.7)
    plt.xlabel('QA Pairs per Entity')
    plt.ylabel('Number of Entities')
    plt.title('Distribution of QA Pairs per Entity')
    plt.axvline(x=np.mean(entity_qa_counts), color='red', linestyle='--', 
                label=f'Mean: {np.mean(entity_qa_counts):.2f}')
    plt.legend()
    plt.tight_layout()
    if output_dir:
        plt.savefig(os.path.join(output_dir, 'qa_per_entity.png'), dpi=300)
        plt.close()
    
    # Plot 5: Entity Relationship Network (for comparative QAs)
    if any(qa.get('reasoning_type') == 'comparative' for qa in dataset['qa_pairs']):
        create_entity_network(dataset, output_dir)

def create_entity_network(dataset, output_dir=None, max_entities=50):
    """Create network visualization for related entities"""
    G = nx.Graph()
    added_entities = set()
    
    # Get comparative QAs
    comp_qas = [qa for qa in dataset['qa_pairs'] 
               if qa.get('reasoning_type') == 'comparative' 
               and 'entity1_id' in qa and 'entity2_id' in qa]
    
    # Add edges between compared entities
    for qa in comp_qas:
        e1_id, e2_id = qa['entity1_id'], qa['entity2_id']
        
        # Skip if we have enough entities
        if len(added_entities) >= max_entities and (e1_id not in added_entities or e2_id not in added_entities):
            continue
        
        # Add nodes
        for e_id in [e1_id, e2_id]:
            if e_id not in G:
                G.add_node(e_id, name=dataset['entity_index'][e_id]['name'])
                added_entities.add(e_id)
        
        # Add or update edge
        if not G.has_edge(e1_id, e2_id):
            G.add_edge(e1_id, e2_id, weight=1)
        else:
            G[e1_id][e2_id]['weight'] += 1
    
    # Only proceed if we have nodes
    if not G.nodes():
        return None
    
    # Draw network
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, seed=42)
    
    # Calculate node sizes
    node_sizes = [100 + len(dataset['entity_index'][n]['qa_ids'])*5 for n in G.nodes()]
    edge_widths = [G[u][v]['weight'] for u, v in G.edges()]
    
    # Draw the network
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', alpha=0.8)
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5, edge_color='gray')
    
    # Use names as labels
    labels = {node: G.nodes[node]['name'] for node in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_weight='bold')
    
    plt.title('Entity Relationship Network')
    plt.axis('off')
    
    if output_dir:
        plt.savefig(os.path.join(output_dir, 'entity_network.png'), dpi=300)
        plt.close()
    
    return G

def generate_summary(stats):
    """Generate markdown summary of dataset stats"""
    summary = [
        f"# ChEBI QA Dataset Summary",
        f"Total QA Pairs: {stats['total_qa_pairs']}",
        f"Total Entities: {stats['total_entities']}",
        f"Average QA Pairs Per Entity: {stats['avg_qa_per_entity']:.2f}",
        f"Average Question Length: {stats['avg_question_length']:.1f} words",
        f"Average Answer Length: {stats['avg_answer_length']:.1f} words",
        "",
        "## Reasoning Types",
    ]
    
    # Add reasoning types
    for rtype, count in sorted(stats['reasoning_types'].items(), key=lambda x: x[1], reverse=True):
        pct = (count / stats['total_qa_pairs']) * 100
        summary.append(f"- {rtype}: {count} ({pct:.1f}%)")
    
    summary.append("\n## Top Reasoning Subtypes")
    
    # Add top reasoning subtypes
    for subtype, count in sorted(stats['reasoning_subtypes'].items(), key=lambda x: x[1], reverse=True)[:10]:
        pct = (count / stats['total_qa_pairs']) * 100
        summary.append(f"- {subtype}: {count} ({pct:.1f}%)")
    
    return "\n".join(summary)

if __name__ == "__main__":
    # Replace with your dataset path
    dataset_path = "/home/matt/Proj/Hermeticav2/paper/comparative_qa_dataset.json"
    output_dir = "dataset_analysis"
    
    analyze_chebi_qa_dataset(dataset_path, output_dir)

/tmp/ipykernel_2234304/3797635268.py:115: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([q_lengths, a_lengths], labels=['Question Length', 'Answer Length'],


✅ Analysis complete! Results saved to dataset_analysis
